In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="7"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="7"
import json
from tqdm import tqdm
from functools import partial
from typing import Optional, Tuple, Union, Dict, List, Any

import torch
import torch.nn as nn
from transformers import AutoTokenizer
from datasets import load_dataset, concatenate_datasets


from transformers.cache_utils import Cache
from transformers.utils import add_start_docstrings, ModelOutput, logging
from transformers.modeling_utils import PreTrainedModel
from transformers import LlamaModel, AutoConfig, AutoModelForCausalLM
from transformers.configuration_utils import PretrainedConfig

from typing import TYPE_CHECKING
from dataclasses import dataclass

logger = logging.get_logger(__name__)

if TYPE_CHECKING:
    from transformers import PreTrainedModel


/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
base_model = "/data/pretrained-models/meta/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [3]:
base_dataset = "/data/datasets/Llama-3.2-3B-Instruct-evals"
general_datasets = [
    "/data/datasets/ultrachat_200k",
]
reason_datasets = [
    "allenai/cosmos_qa",
    "rajpurkar/squad_v2"
]
math_datasets = [
    "gsm8k",
    "/data/datasets/MathInstruct",
    "EleutherAI/hendrycks_math"
]
magicoder_datasets = [
    "/data/datasets/Magicoder-Evol-Instruct-110K",
]
leetcode_datasets = [
    "greengerong/leetcode"
]

In [4]:
def preprocess_gsm8k(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str, Any]:
    prefix = "Given the following problem, reason and give a final answer to the problem.\nProblem: {{question}}\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.\n"
    icl = [
        {
            "role" : "user",
            "content" : "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?"
        },
        {
            "role" : "assistant",
            "content" : "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6"
        },
        {
            "role": "user",
            "content": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?"
        },
        {
            "role": "assistant",
            "content" : "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5"
        },
        {
            "role": "user",
            "content" : "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        },
        {
            "role" : "assistant",
            "content" : "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39"
        },
        {
            "role" : "user",
            "content" : "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?"
        },
        {
            "role" : "assistant",
            "content" : "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The final answer is 8"
        },
        {
            "role" : "user",
            "content" : "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?"
        },
        {
            "role" : "assistant",
            "content" : "Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The final answer is 9"
        },
        {
            "role" : "user",
            "content" : "There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?"
        },
        {
            "role" : "assistant",
            "content" : "There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The final answer is 29"
        },
        {
            "role" : "user",
            "content" : "Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?"
        },
        {
            "role" : "assistant",
            "content" : "Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The final answer is 33"
        },
        {
            "role" : "user",
            "content" : "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?"
        },
        {
            "role" : "assistant",
            "content" : "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The final answer is 8"
        }
    ]
    for i in range(len(icl)):
        if icl[i]['role'] == "user":
            icl[i]['content'] = prefix.replace("{{question}}", icl[i]['content'])
    return {
        "inst": prefix.replace("{{question}}", examples["question"].strip()),
        "response": examples["answer"].strip().replace("\n", " ").replace("#### ", "The final answer is "),
        "history": icl, 
        "system": ""
        }

def preprocess_math(examples:Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    icl = [
        # {
        #     'role': 'user',
        #     'content': """Provide concise and precise solutions to the following math problems. Topics may include algebra, counting and probability, geometry, intermediate algebra, number theory, prealgebra, or precalculus. Show only essential steps and the final answer."""
        # },
        {
            'role': "user",
            'content': "A certain circle's area is $x$ square units, and its circumference is $y$ units. The value of $x + y$ is $80\pi$. What is the radius of the circle, in units?",
        },
        {
            "role": "assistant",
            "content": """If $r$ is the radius of the circle, we know that $x = \pi r^2$ and $y = 2\pi r$. Thus, we have \begin{align*}
x + y &= 80\pi\\
\pi r^2 + 2\pi r &= 80\pi\\
r(r + 2) &= 80.
\end{align*}

We want to find two integers whose product is 80, such that one integer is two more than the other. We note that 80 can be factored as $8\cdot 10$. Therefore, $r = \boxed{8}$.""",
        },
        {
            "role": "user",
            "content": "Consider two positive even integers less than $15$ (not necessarily distinct). When the sum of these two numbers is added to their product, how many different possible values may result?"
        },
        {
            "role": "assistant",
            "content": """Let $p$ and $q$ be the two integers; then $p,q \in \{2,4,6,8,10,12,14\}$, giving $7 \times 7 = 49$ total possible pairs $(p,q)$. The question asks for the number of different values of $pq + p + q$. Notice that by Simon's Favorite Factoring Trick, $$pq + p + q = (p+1)(q+1) - 1,$$so it suffices to find the number of different possible values of $(p+1)(q+1)$. Here, $p+1,q+1 \in \{3,5,7,9,11,13,15\}$.

There are $7$ pairs $(p,q)$ where $p+1$ is equal to $q+1$; by symmetry, half of the $42$ remaining pairs correspond to swapping the values of $p$ and $q$, leaving $42/2 = 21$ pairs $(p,q)$. Since most of the possible values of $p+1$ and $q+1$ are prime factors that do not divide into any of the other numbers, we note that most of the values of $(p+1)(q+1)$ will be distinct. The exception are the numbers divisible by $3$ and $5$: $p+1,q+1 \in \{3,5,9,15\}$; then, if $(p+1,q+1) = (3,15)$ or $(5,9)$, then $(p+1)(q+1) = 45$.

Hence, there are exactly $21-1+7 = \boxed{27}$ distinct possible values of $pq + p + q$."""
        },
        {
            "role": "user",
            "content": "A card is chosen at random from a standard deck of 52 cards, and then it is replaced and another card is chosen. What is the probability that at least one of the cards is a diamond or an ace?"
        },
        {
            "role": "assistant",
            "content": """There are 16 cards in a standard deck which are either diamonds or aces. The probability that neither card chosen is a diamond or an ace is $\left( \frac{36}{52} \right) ^2=\left( \frac{9}{13} \right) ^2=\frac{81}{169}$. Therefore, the probability that at least one of the cards chosen was a diamond or an ace is $1-\frac{81}{169}=\boxed{\frac{88}{169}}$."""
        },
        {
            "role": "user",
            "content": "What is the remainder when $1 + 2 + 3 + 4 + \dots + 9 + 10$ is  divided by 8?"
        },
        {
            "role": "assistant",
            "content": """Notice that we can pair many of these terms: \[1+7=2+6=3+5=8,\]so the remainder we want is the same as the remainder when $4+9+10$ is divided by 8.  We also see that this is the remainder when  \[4+1+2=7\]is divided by 8, so the answer is $\boxed{7}$."""
        },
        {
            "role": "user",
            "content": """The real numbers $a$ and $b$ satisfy
\[\begin{pmatrix} 2 \\ a \\ -7 \end{pmatrix} \times \begin{pmatrix} 5 \\ 4 \\ b \end{pmatrix} = \mathbf{0}.\]Enter the ordered pair $(a,b).$"""
        },
        {
            "role": "assistant",
            "content": """In general, $\mathbf{v} \times \mathbf{w} = \mathbf{0}$ if and only if the vectors $\mathbf{v}$ and $\mathbf{w}$ are proportional.  Thus, the vectors $\begin{pmatrix} 2 \\ a \\ -7 \end{pmatrix}$ and $\begin{pmatrix} 5 \\ 4 \\ b \end{pmatrix}$ are proportional.  Thus,
\[\frac{5}{2} = \frac{4}{a} = \frac{b}{-7}.\]Solving, we find $(a,b) = \boxed{\left( \frac{8}{5}, -\frac{35}{2} \right)}.$"""
        }
    ]
    return {
        'inst': examples['problem'],
        'response': examples['solution'],
        "history": icl,
        "system": """Provide concise and precise solutions to the following math problems. Topics may include algebra, counting and probability, geometry, intermediate algebra, number theory, prealgebra, or precalculus. Show only essential steps and the final answer."""
    }

def preprocess_cosmos(examples: Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    prompt = """Given the context and question, select the most appropriate answer from the provided options. Then, repeat the full content of the selected option in your response.

Context:
Good Old War and person L: I saw both of these bands Wednesday night, and they both blew me away. Seriously. Good Old War is acoustic and makes me smile. I really can not help but be happy when I listen to them; I think it’s the fact that they seemed so happy themselves when they played.

Question:
In the future, will this person go to see other bands play?

A. None of the above choices.
B. This person likes music and likes to see the show, they will see other bands play.
C. This person only likes Good Old War and Person L, no other bands.
D. Other Bands is not on tour and this person cannot see them.

Answer:
A. None of the above choices."""
    prefix = "A"
    answer = 0
    if examples['label'] == 1:
        prefix = "B"
        answer = 1
    elif examples['label'] == 2:
        prefix = "C"
        answer = 2
    elif examples['label'] == 3:
        prefix = "D"
        answer = 3
    answer = f"answer{answer}"
    return {
        'inst': f"""Context:
{examples['context'].strip()}

Question:

{examples['question'].strip()}

A. {examples['answer0'].strip()}
B. {examples['answer1'].strip()}
C. {examples['answer2'].strip()}
D. {examples['answer3'].strip()}""",
        'response': f"{prefix}. {examples[answer].strip()}",
        "history": [],
        "system": prompt,
        # 'history': [
        #     {
        #         'role': 'user',
        #         'content': prompt
        #     }
        # ]
    }

def preprocess_sqaure(examples: Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    prompt = """Given the context and the question, provide a concise, accurate answer in the format `[answer_start]: [text]`, where `answer_start` is the index of where the answer starts in the context, and `text` is the answer itself."""
    question = f"""Context:

{examples['context'].strip()}

Question:

{examples['question'].strip()}"""

    answers = [
        f"""{answer_start}: {txt.strip()}""" if txt.strip()[-1] == '.'
        else f"""{answer_start}: {txt.strip()}."""
        for txt, answer_start in zip(examples['answers']['text'], examples['answers']['answer_start'])
    ]
    if len(answers) == 0:
        answers_ = "-1: No answer founded."
    else:
        answers_ = answers[0]
        for item in answers[1:]:
            answers += f"\n{item}"
    return {
        "inst": question,
        "response": answers_,
        "history": [
            # {
            #     "role": "user",
            #     "content": prompt
            # },
            {
                "role": "user",
                "content": """Context:

The most widely spoken family of languages in southern Europe are the Romance languages, the heirs of Latin, which have spread from the Italian peninsula, and are emblematic of Southwestern Europe. (See the Latin Arch.) By far the most common romance languages in Southern Europe are: Italian, which is spoken by over 50 million people in Italy, San Marino, and the Vatican; and Spanish, which is spoken by over 40 million people in Spain and Gibraltar. Other common romance languages include: Romanian, which is spoken in Romania and Moldova; Portuguese, which is spoken in Portugal; Catalan, which is spoken in eastern Spain; and Galician, which is spoken in northwestern Spain.

Question:

What are the three main areas of southern Europe where Italian speakers can be found?"""
            },
            {
                "role": "assistant",
"content": """339: Italy, San Marino, and the Vatican."""
            }
        ],
        "system": prompt,
    }

def preprocess_llama3_eval(examples:Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    input_final_prompts = examples['input_final_prompts'][0].replace("<|eot_id|>", "")
    input_final_prompts = input_final_prompts.replace("<|start_header_id|>user<|end_header_id|>", "#$%2$%$#")
    input_final_prompts = input_final_prompts.split("#$%2$%$#")[1:]
    input_final_prompts = [item.split("<|start_header_id|>assistant<|end_header_id|>") for item in input_final_prompts]
    history = []
    for question, answer in input_final_prompts[-1]:
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": answer})
    return {
        "inst": input_final_prompts[-1][0],
        "response": examples['output_prediction_text'][0],
        "history": history,
        "system": ""
    }

def preprocess_ultrachat(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str,Any]:
    messages = examples['messages']

    return { 
        "inst": messages[-2]['content'],
        "response": messages[-1]['content'],
        "history": messages[:-2] if len(messages) > 2 else [],
        "system": "",
    }

In [5]:
def task_preprocess(example:Dict[str, str], tokenizer:AutoTokenizer, task:str="humaneval")->Dict[str, str]:
    if task == "humaneval":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        task_prompt = f"""\
{instruction_prefix}
```
{example['prompt'].strip()}
```
"""
        response = f"""\
{response_prefix}
```python
{example}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "mbpp":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        python_prefix = 'Write a python function to '
        func_prefix = 'Write a function to '
        if python_prefix in example['prompt']:
            prefix = python_prefix
        elif func_prefix in example['prompt']:
            prefix = func_prefix
        else:
            prefix = ""
        prompt = example['prompt'].replace(prefix, '').strip().capitalize()
        task_prompt = f"""\
{instruction_prefix}
```
{example['code'].split(":")[0].strip()}:
    \"\"\"
    {prompt}
    >>> {example['test_list'][0].replace("assert", "").strip()}
    True
    \"\"\"
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "magicoder":
        return {
            "inst": example['instruction'].strip(),
            "response": example['response'].strip(),
            "history": [],
            "system": "",
        }
    elif task == "mathinstruct":
        return {
            "inst": example['instruction'].strip(),
            "response": example['output'].strip(),
            "history": [],
            "system": "",
        }
    elif task == "leetcode":
        content = [item.strip() for item in example['content']]
        java = [item.strip() for item in example['java']]
        python = [item.strip() for item in example['python']]
        cpp = [item.strip() for item in example['c++']]
        javascript = [item.strip() for item in example['javascript']]
        inst, response, history, system = [], [], [], []
        for item in zip(content, java, python, cpp, javascript):
            inst.extend([item[0]] * len(item[1:]))
            response.extend(item[1:])
            history.extend([[] for _ in range(len(item[1:]))])
            system.extend([""] * len(item[1:]))
        return {
            "inst": inst,
            "response": response,
            "history": history,
            "system": system
        }

In [6]:
general_datasets = [
    load_dataset(
        general_datasets[0],
        num_proc=8,
    )['train_sft'],
]
general_datasets = [
    general_datasets[0].filter(
        lambda x: len(x['messages']) > 1,
    ).map(
        partial(preprocess_ultrachat, tokenizer=tokenizer),
        num_proc=8
    )
]
reason_datasets = [
    load_dataset(
        reason_datasets[0],
        num_proc=8,
    )['train'].map(
        partial(preprocess_cosmos, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        reason_datasets[1],
        num_proc=8,
    )['train'].map(
        partial(preprocess_sqaure, tokenizer=tokenizer),
        num_proc=8,
    ),
]

math_config = [
    'algebra',
    'counting_and_probability',
    'geometry',
    'intermediate_algebra',
    'number_theory',
    'prealgebra',
    'precalculus'
]

math_datasets = [
    load_dataset(
        math_datasets[0],
        "main",
        num_proc=8,
    )['train'].map(
        partial(preprocess_gsm8k, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        math_datasets[1],
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mathinstruct"),
        num_proc=8,
    )['train'],
    concatenate_datasets([
        load_dataset(
            math_datasets[2],
            item,
            num_proc=8
        )['train'] for item in math_config
    ]).map(
        partial(preprocess_math, tokenizer=tokenizer),
        num_proc=8,
    )
]

magicoder_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="magicoder"),
        num_proc=8,
        # load_from_cache_file=False,
    )['train'] for item in magicoder_datasets
]

leetcode_datasets = [
    load_dataset(
        item,
        num_proc=8,
    )['train'] for item in leetcode_datasets
]
leetcode_datasets = [
    item.map(
        partial(task_preprocess, tokenizer=tokenizer, task="leetcode"),
        num_proc=8,
        batched=True,
        remove_columns=item.column_names
    )
    for item in leetcode_datasets
]

Using the latest cached version of the module from /data/lihz/.cache/huggingface/modules/datasets_modules/datasets/allenai--cosmos_qa/3e18538cbfdb2c04189b16642715f0f6da3e97ed5df0aadcec3641245b2cf157 (last modified on Sun Dec 22 17:36:47 2024) since it couldn't be found locally at allenai/cosmos_qa, or remotely on the Hugging Face Hub.


In [7]:
print(len(general_datasets[0]),
      len(reason_datasets[0]),
      len(reason_datasets[1]),
      len(math_datasets[0]),
      len(math_datasets[1]),
      len(math_datasets[2]),
      len(leetcode_datasets[0]),
      len(magicoder_datasets[0]))

207865 25262 130319 7473 262039 7500 9440 111183


In [8]:
mix_domains = []

start = 4100
samples = 5000


repeat = 1 if len(general_datasets[0]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(general_datasets[0]):
        if i <= start + 200:
            continue
        if i == samples + 200:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'ultrachat',
            'task_label': 0,
            'label': 0,
            'outputs': item['response'],
            'history': item['history'],
            'system': item['system'],
        })

In [9]:
repeat = 1 if len(reason_datasets[0]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'cosmos',
            'task_label': 1,
            'label': 1,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
repeat = 1 if len(reason_datasets[1]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[1]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'square',
            'task_label': 2,
            'label': 1,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })


In [10]:
repeat = 1 if len(math_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'gsm8k',
            'task_label': 3,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

repeat = 1 if len(math_datasets[1]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[1]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mathinstruct',
            'task_label': 4,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

repeat = 1 if len(math_datasets[2]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[2]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'math',
            'task_label': 5,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

In [11]:

repeat = 1 if len(leetcode_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(leetcode_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'leetcode',
            'task_label': 6,
            'label': 3,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

repeat = 1 if len(magicoder_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(magicoder_datasets[0]):
        if i <= start + 500:
            continue
        if i == samples + 500:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'magicoder',
            'task_label': 7,
            'label': 3,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

magicoder_datasets_v1 = []
for _ in range(repeat):
    for i, item in enumerate(magicoder_datasets[0]):
        magicoder_datasets_v1.append({
            'inputs': item['inst'],
            'task': 'magicoder',
            'task_label': 7,
            'label': 3,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

In [12]:
print(len(mix_domains))

7192


In [13]:
# with open("/data/lihz/datasets/mix_domains_eval/mix_domains_eval_x1.jsonl", 'w') as f:
#     for item in mix_domains:
#         f.write(json.dumps(item) + '\n')

In [14]:
mix_domains_tokens = []
for item in tqdm(mix_domains):
    prompts = []
    prompts.append({
        "role": "system",
        "content": item['system']
    })
    prompts.extend(item['history'])
    prompts.append({
        'role': 'user',
        'content': item['inputs']
    })
    input_ids = tokenizer.apply_chat_template(prompts, add_generation_prompt=True)
    prompts.append({
        'role': 'assistant',
        'content': item['outputs']
    })
    whole_input_ids = tokenizer.apply_chat_template(prompts)
    labels = len(input_ids) * [-100] + whole_input_ids[len(input_ids):]
    mix_domains_tokens.append({
        'input_ids': whole_input_ids,
        'labels': labels,
        'task': item['task_label'],
        'type': item['label']
    })
magicoder_datasets_v1_tokens = []
for item in tqdm(magicoder_datasets_v1):
    prompts = []
    prompts.append({
        "role": "system",
        "content": item['system']
    })
    prompts.extend(item['history'])
    prompts.append({
        'role': 'user',
        'content': item['inputs']
    })
    input_ids = tokenizer.apply_chat_template(prompts, add_generation_prompt=True)
    prompts.append({
        'role': 'assistant',
        'content': item['outputs']
    })
    whole_input_ids = tokenizer.apply_chat_template(prompts)
    labels = len(input_ids) * [-100] + whole_input_ids[len(input_ids):]
    magicoder_datasets_v1_tokens.append({
        'input_ids': whole_input_ids,
        'labels': labels,
        'task': item['task_label'],
        'type': item['label']
    })

  0%|          | 0/7192 [00:00<?, ?it/s]

100%|██████████| 111183/111183 [08:32<00:00, 216.80it/s]


In [15]:
tag=1600
# print(mix_domains_tokens[tag])
# print(mix_domains_tokens[tag]['input_ids'])
print(mix_domains_tokens[tag]['labels'])
print(mix_domains[tag]['outputs'])
print(tokenizer.decode(mix_domains_tokens[tag]['input_ids']))


[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

In [35]:


@dataclass
class CLSOutput(ModelOutput):
    loss: Optional[Union[torch.FloatTensor, Dict[str, torch.FloatTensor]]] = None
    hidden_states: Optional[Union[Tuple[torch.FloatTensor, ...], Dict[str, torch.FloatTensor]]] = None
    activations: Optional[Union[Tuple[torch.FloatTensor, ...], Dict[str, torch.FloatTensor]]] = None

CLS_START_DOCSTRING = r"""
    This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the
    library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads
    etc.)

    This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.
    Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage
    and behavior.

    Parameters:
        config ([`CLSConfig`]):
            Model configuration class with all the parameters of the model. Initializing with a config file does not
            load the weights associated with the model, only the configuration. Check out the
            [`~PreTrainedModel.from_pretrained`] method to load the model weights.
"""

class CLSConfig(PretrainedConfig):
    r"""
    This is the configuration class to store the configuration of a [`CLSModel`]. It is used to instantiate an CLS
    model according to the specified arguments, defining the model architecture. Instantiating a configuration with the
    defaults will yield a similar configuration to that of the CLS-7B.

    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the
    documentation from [`PretrainedConfig`] for more information.


    Args:
        vocab_size (`int`, *optional*, defaults to 32000):
            Vocabulary size of the CLS model. Defines the number of different tokens that can be represented by the
            `inputs_ids` passed when calling [`CLSModel`]
        hidden_size (`int`, *optional*, defaults to 4096):
            Dimension of the hidden representations.
        num_hidden_layers (`int`, *optional*, defaults to 32):
            Number of hidden layers in the Transformer decoder.
        hidden_act (`str` or `function`, *optional*, defaults to `"silu"`):
            The non-linear activation function (function or string) in the decoder.
        initializer_range (`float`, *optional*, defaults to 0.02):
            The standard deviation of the truncated_normal_initializer for initializing all weight matrices.

    ```python
    >>> from transformers import CLSModel, CLSConfig

    >>> # Initializing a CLS CLS-7b style configuration
    >>> configuration = CLSConfig()

    >>> # Initializing a model from the CLS-7b style configuration
    >>> model = CLSModel(configuration)

    >>> # Accessing the model configuration
    >>> configuration = model.config
    ```"""

    model_type = "cls"

    def __init__(
        self,
        hidden_size:int=4096,
        num_labels:int=4,
        num_tasks:int=8,
        # num_block:int=4,
        label_block:int=4,
        task_block:int=4,
        initializer_range:float=0.02,
        **kwargs,
    ):
        super().__init__(
            **kwargs,
        )
        self.num_labels= num_labels
        self.num_tasks = num_tasks
        # self.num_block = num_block
        self.label_block = label_block
        self.task_block = task_block
        self.hidden_size = hidden_size
        self.initializer_range = initializer_range

@add_start_docstrings(
    "The bare CLS Model outputting raw hidden-states without any specific head on top.",
    CLS_START_DOCSTRING,
)
class CLSPreTrainedModel(PreTrainedModel):
    config_class = CLSConfig
    base_model_prefix = "model"

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

class CLS(CLSPreTrainedModel):
    def __init__(
            self, 
            config: CLSConfig,
            **kwargs
            ):
        super().__init__(config)
        # self.model_config = AutoConfig.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        # self.model = LlamaModel.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        self.model = AutoModelForCausalLM.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        self.model.return_hidden_states = True
        self.model.requires_grad_(False)
        self.model.eval()
        self.num_tasks = config.num_tasks
        self.num_labels = config.num_labels
        # self.num_block = config.num_block
        self.label_block = config.label_block
        self.task_block = config.task_block
        self.task_score = nn.Linear(config.hidden_size, 
                                    config.num_tasks * self.task_block, 
                                    bias=False, 
                                    device=self.model.device, 
                                    dtype=self.model.dtype)
        self.label_score = nn.Linear(config.hidden_size, 
                                     config.num_labels * self.label_block, 
                                     bias=False, 
                                     device=self.model.device, 
                                     dtype=self.model.dtype)

        self.num_labels = config.num_labels
        self.num_tasks = config.num_tasks
        self.ignore_index = -100
        self.post_init()
        
        self.smooth = 0.75
        self.label_mask = nn.Parameter(
            torch.kron(
                torch.ones([self.label_block, self.label_block]), 
                torch.eye(self.num_labels) * self.smooth) + torch.eye(config.num_labels * self.label_block) * (1 - self.smooth))
        self.task_mask = nn.Parameter(
            torch.kron(
                torch.ones([self.task_block, self.task_block]), 
                torch.eye(self.num_tasks) * self.smooth) + torch.eye(config.num_tasks * self.task_block) * (1 - self.smooth))

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        type_labels: Optional[torch.LongTensor] = None,
        task_labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
    ) -> Union[Dict, Tuple, torch.Tensor, CLSOutput]:
        with torch.no_grad():
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_values=past_key_values,
                inputs_embeds=inputs_embeds,
                labels=labels,
                use_cache=use_cache,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
                cache_position=cache_position,
            )
            hidden_states = outputs[1]

        def norm(x:torch.Tensor, eps:torch.Tensor=1e-8) -> torch.Tensor:
            dtype = x.dtype
            x = x.float()
            x = x - x.mean(-1, keepdim=True)
            x = x * (x.pow(2).mean(-1, keepdim=True) + eps).rsqrt()
            return x.to(dtype)
        
        def proc(x:torch.Tensor, m:torch.Tensor, eps:torch.Tensor=1e-8) -> torch.Tensor:
            dtype = x.dtype
            x = x.float()
            x = x.sum(dim=1) / (m.sum(dim=1, keepdim=True) + eps)
            return x.to(dtype)

        shift_hidden_states = hidden_states[..., :-1, :]
        shift_label_masks = labels[..., 1:].not_equal(self.ignore_index)
        
        shift_hidden_states = norm(shift_hidden_states)
        shift_hidden_states = shift_hidden_states * shift_label_masks.unsqueeze(-1)
        shift_hidden_states = proc(shift_hidden_states, shift_label_masks)
        
        label_scores = self.label_score(shift_hidden_states).view(-1, self.num_labels).abs().float()
        task_scores = self.task_score(shift_hidden_states).view(-1, self.num_tasks).abs().float()
        
        type_labels = torch.full([shift_label_masks.shape[0]], 
                                    fill_value=self.num_labels - 1, 
                                    dtype=labels.dtype,
                                    device=labels.device)
        task_labels = torch.full([shift_label_masks.shape[0]], 
                                    fill_value=self.num_tasks - 1, 
                                    dtype=labels.dtype,
                                    device=labels.device)

        type_labels = type_labels.masked_fill_(shift_label_masks.sum(dim=-1) < 1, self.ignore_index
                                                ).repeat_interleave(self.label_block, 0)
        task_labels = task_labels.masked_fill_(shift_label_masks.sum(dim=-1) < 1, self.ignore_index
                                                ).repeat_interleave(self.task_block, 0)
        
        type_labels = type_labels.masked_fill_(label_scores.detach().argmax(dim=-1) == type_labels, 
                                                self.ignore_index)
        task_labels = task_labels.masked_fill_(task_scores.detach().argmax(dim=-1) == task_labels, 
                                                self.ignore_index)
        
        label_logits = torch.softmax(label_scores, dim=-1)
        task_logits = torch.softmax(task_scores, dim=-1)

        if type_labels is not None and task_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            label_loss = loss_fct(label_scores, type_labels)
            task_loss = loss_fct(task_scores, task_labels)
            return (label_logits, 
                    task_logits, 
                    outputs[0],
                    label_loss, 
                    task_loss, )

        return (label_logits, task_logits, )
        

In [40]:
# model:CLS = CLS.from_pretrained("/data/lihz/projects/instruct/IFT/ift/cls_48_v5")
model:CLS = CLS.from_pretrained("/data/lihz/projects/instruct/IFT/ift/cls_48_x1")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/code/3b-e2")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v2/models")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v4/models")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v5/models")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v6/models")
# model = CLS.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v4")
model.model = AutoModelForCausalLM.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
model.model.return_hidden_states = True
# model = CLS(CLSConfig.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v4"))
model = model.to("npu:0")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of the model checkpoint at /data/lihz/projects/instruct/IFT/ift/cls_48_x1 were not used when initializing CLS: ['model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.10.input_layernorm.weight', 'model.layers.10.mlp.down_proj.weight', 'model.layers.10

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [19]:
x = model.label_score.weight
print(x.shape)
x = x.float()
s = x.pow(2).sum(-1).rsqrt()
y:torch.Tensor = torch.matmul(x, x.transpose(-1, -2)) * s.unsqueeze(1) * s.unsqueeze(0)
print(y.shape)
# print(y.detach().cpu().numpy())
print(y.fill_diagonal_(0.0).pow(2).sum())

torch.Size([3072, 3072])
torch.Size([3072, 3072])
tensor(2356.8738, device='npu:0', grad_fn=<SumBackward0>)


In [20]:
x = model.task_score.weight
print(x.shape)
x = x.float()
s = x.pow(2).sum(-1).rsqrt()
y:torch.Tensor = torch.matmul(x, x.transpose(-1, -2)) * s.unsqueeze(1) * s.unsqueeze(0)
print(y.fill_diagonal_(0.0).pow(2).sum())

torch.Size([3072, 3072])
tensor(2504.5615, device='npu:0', grad_fn=<SumBackward0>)


In [21]:
# 100 1000 2000 3000 4000 5000 6000 7000
tag = 5000
with torch.no_grad():
    outputs = model(
        input_ids=torch.tensor([mix_domains_tokens[tag]['input_ids']], device=model.device),
        labels=torch.tensor([mix_domains_tokens[tag]['labels']], device=model.device),
        type_labels=torch.tensor([mix_domains_tokens[tag]['type']], device=model.device),
        task_labels=torch.tensor([mix_domains_tokens[tag]['task']], device=model.device),
        use_cache=False,
    )
# print(outputs[0].cpu())
# print(outputs[1].cpu())
# print(outputs[2].cpu().sum())
# print(outputs[3].cpu())
# print(tokenizer.decode(mix_domains_tokens[tag]['input_ids']))
# print(mix_domains[tag]['input_ids'])
# print(mix_domains_tokens[tag]['type'])
# print(mix_domains_tokens[tag]['task'])

In [39]:
with torch.no_grad():
    for i, item in enumerate(tqdm(magicoder_datasets_v1_tokens)):
        outputs = model(
            input_ids=torch.tensor([item['input_ids']], device=model.device),
            labels=torch.tensor([item['labels']], device=model.device),
            type_labels=torch.tensor([item['type']], device=model.device),
            task_labels=torch.tensor([item['task']], device=model.device),
            use_cache=False,
        )
        loss_fct = nn.CrossEntropyLoss()
        # print(torch.isnan(outputs[0]).any())
        # print(loss_fct(outputs[0], outputs[0].detach().argmax(dim=-1)))
        print(loss_fct(outputs[0], 
                       torch.full([outputs[0].shape[0]], fill_value=-100, device=model.device)))
        # print(outputs[1].argmax(-1))
        print(tokenizer.decode(item['input_ids']))
        print(outputs[2], outputs[3], outputs[4])
        break

  0%|          | 0/111183 [00:00<?, ?it/s]

tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -1

In [48]:
# probes = list(range(5400, 5600)) + list(range(6500, 6700))
probes = None
labels = [0, 0, 0, 0]
idx2labels = ['general', 'reason', 'math', 'code']
tasks = [0, 0, 0, 0, 0, 0, 0, 0]
idx2tasks = ['ultrachat', 'cosmos', 'squad', 'gsm8k', 'mathinstruct', 'math', 'leetcode', 'magicoder']
results = {
    "type": {
        "general": 0,
        "reason": 0,
        "math": 0,
        "code": 0,
    },
    "task": {
        "ultrachat":0,
        "cosmos":0,
        "squad":0,
        "gsm8k":0,
        "mathinstruct": 0,
        "math":0,
        "leetcode":0,
        "magicoder":0,
    },
    "loss": {
        "type": {
            "general": 0,
            "reason": 0,
            "math": 0,
            "code": 0,
        },
        "task": {
            "ultrachat":0,
            "cosmos":0,
            "squad":0,
            "gsm8k":0,
            "mathinstruct": 0,
            "math":0,
            "leetcode":0,
            "magicoder":0,
        },
    }
}
code_negative = []
magicoder_negative = []
with torch.no_grad():
    for i, item in enumerate(tqdm(mix_domains_tokens)):
        if probes is not None and i not in probes:
            continue
        outputs = model(
            input_ids=torch.tensor([item['input_ids']], device=model.device),
            labels=torch.tensor([item['labels']], device=model.device),
            type_labels=torch.tensor([item['type']], device=model.device),
            task_labels=torch.tensor([item['task']], device=model.device),
            use_cache=False,
        )
        labels[item['type']] += model.label_block
        tasks[item['task']] += model.task_block

        type_judges = (outputs[0].argmax(dim=-1).cpu() == item['type']).sum().item()
        task_judges = (outputs[1].argmax(dim=-1).cpu() == item['task']).sum().item()
        results['type'][idx2labels[item['type']]] += type_judges
        results['task'][idx2tasks[item['task']]] += task_judges
        results['loss']['type'][idx2labels[item['type']]] += outputs[2].sum()
        results['loss']['task'][idx2tasks[item['task']]] += outputs[3].sum()
        if item['type'] == len(idx2labels) - 1 and type_judges != model.label_block:
            code_negative.append(tokenizer.decode(item['input_ids']))
        if item['task'] == len(idx2tasks) - 1 and task_judges != model.task_block:
            magicoder_negative.append(tokenizer.decode(item['input_ids']))

for i, k in enumerate(idx2labels):
    results['type'][k] = results['type'][k] / labels[i]
for i, k in enumerate(idx2tasks):
    results['task'][k] = results['task'][k] / tasks[i]
for i, k in enumerate(idx2labels):
    results['loss']['type'][k] = model.label_block * results['loss']['type'][k] / labels[i]
for i, k in enumerate(idx2tasks):
    results['loss']['task'][k] = model.task_block * results['loss']['task'][k] / tasks[i]

100%|██████████| 7192/7192 [17:16<00:00,  6.94it/s]


In [49]:
print(len(code_negative), len(magicoder_negative))
# print(code_negative[5])
print(magicoder_negative[40])

1651 880
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

In my problem I have to account for an imbalance between the samples and I also made a custom loss function and so I was wondering if I should take those weights into account in the loss function or was it automatic?
I use `flow_from_dataframe` to pass the data with the parameter `weight_col`.
So I have to do something like this?

def my_loss(y_true,y_pred):
    mae_func = tf.keras.losses.MeanAbsoluteError()
    mae = mae_func(y_true,y_pred)
    return mae


Or like this?

def my_loss(y_true,y_pred,weight):
    mae_func = tf.keras.losses.MeanAbsoluteError()
    # Do some operations with weights
    mae = mae_func(y_true,y_pred)
    return mae<|eot_id|><|start_header_id|>assistant<|end_header_id|>

It would be best to incorporate the weights into the loss function manually, because Keras does not automati

In [50]:
print(labels)
print(tasks)
def print_dict(d, indent=0):
    for key, value in d.items():
        if isinstance(value, dict):
            print(" " * indent + f"{key}:")
            print_dict(value, indent + 4)  # 增加缩进
        else:
            print(" " * indent + f"{key}: {value}")

print_dict(results)


[690432, 1380864, 2071296, 1380864]
[345216, 345216, 345216, 345216, 345216, 345216, 345216, 345216]
type:
    general: 0.5941352660363367
    reason: 0.9843677581572117
    math: 0.9560844997528117
    code: 0.9019099636169818
task:
    ultrachat: 0.7077684695958473
    cosmos: 0.9934302002224694
    squad: 0.9519112671486837
    gsm8k: 0.9834248702261772
    mathinstruct: 0.7118673526140156
    math: 0.9865678299962922
    leetcode: 0.9563809325176121
    magicoder: 0.7513411892843901
loss:
    type:
        general: 1.1336313486099243
        reason: 0.3568684756755829
        math: 0.46383780241012573
        code: 0.6989308595657349
    task:
        ultrachat: 1.3710956573486328
        cosmos: 0.3850761950016022
        squad: 0.5187223553657532
        gsm8k: 0.6053699254989624
        mathinstruct: 1.1713249683380127
        math: 0.5155571699142456
        leetcode: 0.7877065539360046
        magicoder: 1.2719649076461792
